### Data Loading and Working Copy

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
current_dir = Path.cwd()
raw_dir = None

for folder in [current_dir, *current_dir.parents]:
    candidate = folder / "data" / "raw"
    if candidate.is_dir():
        raw_dir = candidate
        break

if raw_dir is None:
    raise FileNotFoundError("Could not locate data/raw folder")

csv_files = list(raw_dir.glob("*.csv"))

if len(csv_files) != 1:
    raise ValueError(
        f"Expected one CSV file, found {len(csv_files)}"
    )

df_raw = pd.read_csv(
    csv_files[0],
    encoding="windows-1252"
)
df_raw.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df_clean = df_raw.copy()

print("Original dataset shape:", df_raw.shape)
print("Working copy shape:", df_clean.shape)

Original dataset shape: (9994, 21)
Working copy shape: (9994, 21)


### Date Type Correction.

In [4]:
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"], format="%m/%d/%Y")

df_clean["Ship Date"] = pd.to_datetime(df_clean["Ship Date"], format="%m/%d/%Y")

print("Date column types:")
display(df_clean[["Order Date", "Ship Date"]].dtypes.to_frame("Data Type"))

print("Missing dates after conversion:")
display(df_clean[["Order Date", "Ship Date"]].isna().sum().to_frame("Missing Values"))

print("Dataset shape:", df_clean.shape)

Date column types:


,Data Type
Order Date,datetime64[us]
Ship Date,datetime64[us]


Missing dates after conversion:


,Missing Values
Order Date,0
Ship Date,0


Dataset shape: (9994, 21)


### Postal Code Standardization.

In [5]:
df_clean["Postal Code"] = df_clean["Postal Code"].astype("string").str.zfill(5)

print("Postal Code data type:", df_clean["Postal Code"].dtype)

print("Postal codes not having 5 characters:", df_clean["Postal Code"].str.len().ne(5).sum())

print("Dataset shape:", df_clean.shape)

display(
    df_clean.loc[df_clean["Postal Code"].str.startswith("0"),["Postal Code", "City", "State"]]
    .drop_duplicates()
    .head(10)
)

Postal Code data type: string
Postal codes not having 5 characters: 0
Dataset shape: (9994, 21)


,Postal Code,City,State
185,06824,Fairfield,Connecticut
197,07090,Westfield,New Jersey
267,07960,Morristown,New Jersey
298,07109,Belleville,New Jersey
306,08701,Lakewood,New Jersey
313,07601,Hackensack,New Jersey
346,01852,Lowell,Massachusetts
366,06040,Manchester,Connecticut
377,02038,Franklin,Massachusetts
395,02886,Warwick,Rhode Island


### Exact Duplicate Record Handling

In [6]:
business_columns = df_clean.columns.drop("Row ID")

rows_before = len(df_clean)

duplicate_mask = df_clean.duplicated(subset=business_columns, keep="first")

print("Extra duplicate records identified:", duplicate_mask.sum())

display(
    df_clean.loc[df_clean.duplicated(subset=business_columns, keep=False)]
    .sort_values(["Order ID", "Product ID", "Row ID"])
)

df_clean = df_clean.drop_duplicates(subset=business_columns, keep="first").copy()

print("Rows before removal:", rows_before)
print("Rows removed:", rows_before - len(df_clean))
print("Rows after removal:", len(df_clean))
print("Remaining duplicate business records:",
      df_clean.duplicated(subset=business_columns).sum())

Extra duplicate records identified: 1


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3405,3406,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
3406,3407,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


Rows before removal: 9994
Rows removed: 1
Rows after removal: 9993
Remaining duplicate business records: 0


### Final Data Quality Validation.

In [7]:
business_columns = df_clean.columns.drop("Row ID")

validation = {
    "Total Rows": len(df_clean),
    "Total Columns": len(df_clean.columns),
    "Missing Values": df_clean.isna().sum().sum(),
    "Duplicate Row IDs": df_clean["Row ID"].duplicated().sum(),
    "Duplicate Business Records": df_clean.duplicated(subset=business_columns).sum(),
    "Invalid Order Dates": df_clean["Order Date"].isna().sum(),
    "Invalid Ship Dates": df_clean["Ship Date"].isna().sum(),
    "Ship Before Order": (df_clean["Ship Date"] < df_clean["Order Date"]).sum(),
    "Invalid Postal Code Length": (df_clean["Postal Code"].str.len().ne(5)).sum(),
    "Invalid Discounts": (~df_clean["Discount"].between(0, 1)).sum(),
    "Invalid Sales": (df_clean["Sales"] <= 0).sum(),
    "Invalid Quantity": (df_clean["Quantity"] <= 0).sum()
}

display(pd.Series(validation).to_frame("Count"))

print("\nFinal Data Types:")
display(df_clean.dtypes.to_frame("Data Type"))

,Count
Total Rows,9993
Total Columns,21
Missing Values,0
Duplicate Row IDs,0
Duplicate Business Records,0
Invalid Order Dates,0
Invalid Ship Dates,0
Ship Before Order,0
Invalid Postal Code Length,0
Invalid Discounts,0



Final Data Types:


,Data Type
Row ID,int64
Order ID,str
Order Date,datetime64[us]
Ship Date,datetime64[us]
Ship Mode,str
Customer ID,str
Customer Name,str
Segment,str
Country,str
City,str


### Export Cleaned Dataset.

In [8]:
processed_dir = raw_dir.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "superstore_cleaned.csv"

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

In [9]:
export_check = pd.read_csv(
    output_path,
    encoding="utf-8-sig",
    dtype={"Postal Code": "string"}
)

print("Reloaded dataset shape:", export_check.shape)

print(
    "Invalid postal codes:",
    export_check["Postal Code"].str.len().ne(5).sum()
)

Reloaded dataset shape: (9993, 21)
Invalid postal codes: 0
